In [1]:
!pip install -q transformers sentencepiece accelerate pandas
from google.colab import files
import pandas as pd

files.upload()

df = pd.read_csv("xed_dev.tsv", sep="\t")

print("Number of examples:", len(df))
print(df[["english_text", "es_text", "emotion_labels"]].head())


Saving xed_dev.tsv to xed_dev.tsv
Number of examples: 954
                                        english_text  \
0                             I'll take care of him.   
1  Hey, you know there was a guy in the game that...   
2                  I don't want any lesson, Shooter.   
3                        What happened to your coat?   
4                        - What was the movie about?   

                                             es_text         emotion_labels  
0                             Yo me encargaré de él.       ["joy", "trust"]  
1  ¿Sabes que había un tipo en el juego que me ac...  ["anger", "surprise"]  
2                      No quiero lecciones, Shooter.              ["anger"]  
3                          ¿Qué le pasó a tu abrigo?           ["surprise"]  
4                  - ¿De qué se trataba la película?       ["anticipation"]  


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "bigscience/mt0-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"

model = model.to(device)
model.eval()

print("Model loaded successfully!")
print("Using:", device)

config.json:   0%|          | 0.00/773 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/430 [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B / 4.31MB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 16.3MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Model loaded successfully!
Using: cpu


In [3]:
def create_zero_shot_prompt(sentence):

    prompt = f"""
Classify the emotions expressed in the following text.

Allowed emotions:
anger, anticipation, disgust, fear,
joy, sadness, surprise, trust.

A sentence may express one or several emotions.

Return only a JSON list containing the English names
of the detected emotions.

Text: {sentence}

Emotions:
"""

    return prompt

In [4]:
sentence = "Estoy muy feliz hoy."

prompt = create_zero_shot_prompt(sentence)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=False
    )

prediction = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Sentence:", sentence)
print("Model prediction:", prediction)

Sentence: Estoy muy feliz hoy.
Model prediction: Emotions: sadness, joy, joy, sadness, joy, sadness, joy, sadness, trust
